In [ ]:
import os

In [ ]:
ld_library_path = os.environ.get("LD_LIBRARY_PATH", "") # Get current LD_LIBRARY_PATH (if set)
conda_lib_path = 'data/conda/envs/swift/lib:/lib/x86_64-linux-gnu'
new_ld_library_path = f"{conda_lib_path}:{ld_library_path}"
os.environ["LD_LIBRARY_PATH"] = new_ld_library_path # Set it

# model

In [ ]:
import sys
sys.path.append('workplace/RA/repo/Qwen2.5-VL/qwen-vl-utils/src/')

from qwen_vl_utils import process_vision_info

In [ ]:
import torch

from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor

In [ ]:
# We recommend enabling flash_attention_2 for better acceleration and memory saving, especially in multi-image and video scenarios.
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct",
    torch_dtype=torch.float16,
    attn_implementation="flash_attention_2",
    device_map="auto",
)

# default processer
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")

# data

In [ ]:
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

In [ ]:
data_dir = '../data/'
synthetic_data_dir = '../synthetic_data/'
os.listdir(data_dir)

In [ ]:
disease_images_df = pd.read_csv(os.path.join(data_dir, 'disease_images.csv'))
disease_images_df.head()

In [ ]:
os.listdir(synthetic_data_dir)

In [ ]:
df = []
for  file_name in os.listdir(os.path.join(synthetic_data_dir, 'fastgan')):
    if file_name.endswith('.csv'):
        cur_df = pd.read_csv(os.path.join(synthetic_data_dir, 'fastgan', file_name))
        cur_df['subdir'] = file_name.split('.')[0]
        df.append(cur_df)
df = pd.concat(df, ignore_index=True)
df

In [ ]:
df.columns

In [ ]:
grouped_df = df.groupby('disease_label_cos')
for abbr, abbr_df in tqdm(grouped_df):
    # sort on column 'cos' 
    abbr_df = abbr_df.sort_values(by='cos', ascending=False)
    abbr_df = abbr_df.reset_index(drop=True)
    for i, row in abbr_df.iterrows():
        if i > 10:
            break
        image_path = os.path.join(synthetic_data_dir, 'fastgan', row['subdir'], 'img', row['image_name'])
        output_file_name = os.path.join( row['subdir'], row['image_name'].split('.')[0] + '.txt')

        text = (
            "You are a professional clinical geneticist.\n\n"
            "Given the facial image, generate a concise diagnostic report focusing on facial features.\n\n"
            "**Instructions:**\n"
            "- Begin with an overall impression paragraph summarizing the patient's facial appearance.\n"
            "- Then, for each relevant facial region, write a single paragraph to describe the facial features.\n"
            "- Finish with a prediction of potentical disease and a diagnostic suggestion or next steps.\n\n"
            "**Facial Diagnostic Report:**\n\n"
            "Overall Impression:\n\n"
            "Left Eye:\n"
            "Right Eye:\n"
            "Nose:\n"
            "Mouth/Lips:\n"
            "Diagnostic Suggestion:\n"
        )
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "image": image_path,  
                    },
                    {
                        "type": "text",
                        "text":  text,
                    }
                ]
            }
        ]
        
        with torch.no_grad():
            # Preparation for inference
            text = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
            image_inputs, video_inputs = process_vision_info(messages)
            inputs = processor(
                text=[text],
                images=image_inputs,
                videos=video_inputs,
                padding=True,
                return_tensors="pt",
            )
            inputs = inputs.to("cuda")
            
            # Inference: Generation of the output
            generated_ids = model.generate(**inputs, max_new_tokens=1024)
            generated_ids_trimmed = [
                out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]

            output_text = processor.batch_decode(
                generated_ids, 
                # generated_ids_trimmed, 
                skip_special_tokens=True, clean_up_tokenization_spaces=False
            )
            answer = output_text[0]
            # write messages to a txt file
            with open(os.path.join('../reports/', 'fastgan', output_file_name), 'w') as f:
                f.write(answer)
    